# P1: Water Quality + Daily Environment Merge

Primary merge per `merge.md` (§2, **P1**). Combines every `02_clean` table that
varies at daily resolution with the EPA water-quality measurements, at the WQ
measurement's native grain (one row per `MonitoringLocationIdentifier` +
`ActivityStartDateTime`).

**Inputs** (all `data/tabular/02_clean/...`):
- `water-quality/epa-wq-clean.csv` — the measurement events (base table)
- `water-quality/epa-stations-clean.csv` — station geography, joined on `MonitoringLocationIdentifier`
- `climate/prism-iowa-climate-clean.csv` — joined directly on `MonitoringLocationIdentifier` (= `station_id`) + date
- `climate/isu-climate-clean.csv` — joined via **haversine nearest-neighbor** spatial match (no shared key with WQ stations) + date
- `streamflow/usgs-iowa-discharge-clean.csv` + `streamflow/usgs-iowa-gauges-clean.csv` — gauges matched via **haversine nearest-neighbor** spatial match, then discharge joined on the matched gauge + date

**PRISM vs. ISU temperature overlap:** both sources report max/min temperature.
PRISM is treated as authoritative (Step 2) since it's sampled directly at the
station coordinate with no spatial-match error; ISU's redundant max/min
temperature and max/min dewpoint columns are dropped in Step 3, keeping only
the fields ISU uniquely contributes (wind, humidity, snow, feels-like,
climatological normals).

**Output:** `data/03a_merge_primary/wq-daily-environment-clean.csv`, one row per WQ measurement event.

In [1]:
import os

import numpy as np
import pandas as pd
import requests
from scipy.spatial import cKDTree

CLEAN = "../../data/tabular/02_clean"
OUT_DIR = "../../data/03a_merge_primary"
OUT_FILE = f"{OUT_DIR}/wq-daily-environment.csv"

EARTH_RADIUS_KM = 6371.0


def nearest_neighbor_match(sources, source_id, target_lookup, target_id, target_lat, target_lon):
    """For each row in `sources` (with lat/lon columns), find the nearest row in
    `target_lookup` by haversine distance. Returns a DataFrame keyed on `source_id`
    with the matched `target_id` and the distance in km."""
    target_coords_rad = np.radians(target_lookup[[target_lat, target_lon]].values)
    tree = cKDTree(target_coords_rad)

    source_coords_rad = np.radians(sources[["LatitudeMeasure", "LongitudeMeasure"]].values)
    chord_dist, indices = tree.query(source_coords_rad, k=1)
    great_circle_km = 2 * EARTH_RADIUS_KM * np.arcsin(np.clip(chord_dist / 2, -1, 1))

    return pd.DataFrame({
        source_id: sources[source_id].values,
        target_id: target_lookup[target_id].iloc[indices].values,
        "distance_km": great_circle_km,
    })

## Step 1: Load water quality measurements + station geography

In [2]:
df_wq = pd.read_csv(
    f"{CLEAN}/water-quality/epa-wq-clean.csv",
    parse_dates=["ActivityStartDateTime"],
    low_memory=False,
)
df_stations = pd.read_csv(f"{CLEAN}/water-quality/epa-stations-clean.csv")

# Daily-resolution join key shared by every environmental table below
df_wq["activity_date"] = df_wq["ActivityStartDateTime"].dt.normalize()

print(f"WQ measurements: {df_wq.shape}")
print(f"Stations:        {df_stations.shape}")
assert not df_wq.duplicated(subset=["MonitoringLocationIdentifier", "ActivityStartDateTime"]).any(), (
    "WQ grain violated: duplicate (station, timestamp) rows"
)

WQ measurements: (48251, 49)
Stations:        (1666, 10)


In [3]:
df = df_wq.merge(df_stations, on="MonitoringLocationIdentifier", how="left")

print(f"Merged shape: {df.shape}")
print(f"Stations matched: {df['LatitudeMeasure'].notna().sum():,} / {len(df):,}")
assert len(df) == len(df_wq), "Station join fanned out WQ rows"
df.head(3)

Merged shape: (48251, 58)
Stations matched: 48,251 / 48,251


,MonitoringLocationIdentifier,ActivityStartDateTime,"Temperature, water_value","Temperature, water_unit",Dissolved oxygen (DO)_value,Dissolved oxygen (DO)_unit,pH_value,pH_unit,Nitrate_value,Nitrate_unit,...,activity_date,OrganizationIdentifier,MonitoringLocationName,MonitoringLocationTypeName,HUCEightDigitCode,LatitudeMeasure,LongitudeMeasure,StateCode,CountyCode,ProviderName
0,11NPSWRD_WQX-HTLN_EFMO_DOUS1,2017-07-18 15:00:00,19.820,deg C,8.75,mg/L,8.330,std units,NaN,NaN,...,2017-07-18,11NPSWRD_WQX,EFMO Dousman Creek Site 1,River/Stream,7060001,43.089707,-91.215905,19,5,STORET
1,11NPSWRD_WQX-HTLN_EFMO_DOUS1,2023-07-19 11:00:00,16.372,deg C,12.24,mg/L,8.215,std units,NaN,NaN,...,2023-07-19,11NPSWRD_WQX,EFMO Dousman Creek Site 1,River/Stream,7060001,43.089707,-91.215905,19,5,STORET
2,11NPSWRD_WQX-HTLN_HEHO_HOOV1,2017-07-17 17:00:00,19.920,deg C,7.62,mg/L,8.195,std units,NaN,NaN,...,2017-07-17,11NPSWRD_WQX,HEHO Hoover Creek,River/Stream,7080206,41.669562,-91.348484,19,31,STORET


## Step 2: PRISM climate — direct join

PRISM samples the climate grid at the exact WQ station coordinate, so `station_id`
in the PRISM table *is* `MonitoringLocationIdentifier` — no spatial matching
needed, just a key + date join.

In [4]:
df_prism = pd.read_csv(f"{CLEAN}/climate/prism-iowa-climate-clean.csv", parse_dates=["date"])

df = df.merge(
    df_prism,
    left_on=["MonitoringLocationIdentifier", "activity_date"],
    right_on=["station_id", "date"],
    how="left",
)
df = df.drop(columns=["station_id", "date"])

print(f"Merged shape: {df.shape}")
print(f"PRISM match rate: {df['prism_tmax_c'].notna().mean():.1%}")
assert len(df) == len(df_wq), "PRISM join fanned out WQ rows"

Merged shape: (48251, 62)
PRISM match rate: 96.6%


## Step 3: ISU/IEM climate — nearest-station spatial match

ISU climate stations have no shared key with WQ stations, so each WQ station is
matched to its nearest ISU/IEM airport station by haversine distance, then that
station's daily record is joined on the matched station + date.

Station coordinates aren't in `isu-climate-clean.csv` itself — they're fetched
from the IEM `IA_ASOS` network GeoJSON endpoint (the same network the download
step pulled station codes from; same fetch approach as the retired
`src/03_merge/epa-climate-merge.ipynb`), filtered down to the station codes
actually present in our climate data.

In [5]:
df_isu = pd.read_csv(f"{CLEAN}/climate/isu-climate-clean.csv", parse_dates=["day"])
isu_stations_in_data = set(df_isu["station"].unique())
print(f"ISU station-days: {df_isu.shape}, {len(isu_stations_in_data)} unique stations")

IEM_URL = "https://mesonet.agron.iastate.edu/geojson/network/IA_ASOS.geojson"
response = requests.get(IEM_URL, timeout=30)
response.raise_for_status()
geojson = response.json()

isu_meta_rows = [
    {
        "climate_station": feature["id"],
        "climate_station_name": feature["properties"].get("sname", ""),
        "isu_lat": feature["geometry"]["coordinates"][1],
        "isu_lon": feature["geometry"]["coordinates"][0],
    }
    for feature in geojson["features"]
]
df_isu_meta = pd.DataFrame(isu_meta_rows)
df_isu_meta = df_isu_meta[df_isu_meta["climate_station"].isin(isu_stations_in_data)].reset_index(drop=True)

print(f"ISU stations with coordinates: {len(df_isu_meta)}")
assert len(df_isu_meta) > 0, "No ISU station coordinates matched the climate data"

ISU station-days: (221559, 20), 62 unique stations


ISU stations with coordinates: 62


In [6]:
isu_match = nearest_neighbor_match(
    sources=df_stations,
    source_id="MonitoringLocationIdentifier",
    target_lookup=df_isu_meta,
    target_id="climate_station",
    target_lat="isu_lat",
    target_lon="isu_lon",
).rename(columns={"distance_km": "distance_to_climate_station_km"})
isu_match = isu_match.merge(
    df_isu_meta[["climate_station", "climate_station_name"]], on="climate_station", how="left"
)

print("Nearest ISU station distance (km):")
print(isu_match["distance_to_climate_station_km"].describe().round(2))

df = df.merge(isu_match, on="MonitoringLocationIdentifier", how="left")
assert len(df) == len(df_wq), "ISU station mapping join fanned out WQ rows"

Nearest ISU station distance (km):
count    1666.00
mean       23.25
std        13.02
min         0.26
25%        13.14
50%        21.79
75%        30.95
max        74.57
Name: distance_to_climate_station_km, dtype: float64


In [7]:
# PRISM (Step 2) already gives max/min temperature and mean dewpoint sampled
# directly at the station coordinate. ISU's max/min temperature and max/min
# dewpoint are dropped here rather than kept alongside PRISM's: PRISM has no
# spatial-match error while ISU is matched to a station ~23 km away on
# average, so the two columns would just be a noisier and a cleaner copy of
# the same signal. Everything else ISU offers (wind, humidity, snow,
# feels-like, climatological normals) has no PRISM equivalent and is kept.
ISU_REDUNDANT_WITH_PRISM = ["max_temp_c", "min_temp_c", "max_dewpoint_c", "min_dewpoint_c"]

isu_value_cols = [
    c for c in df_isu.columns
    if c not in ("station", "day") and c not in ISU_REDUNDANT_WITH_PRISM
]
df_isu_renamed = df_isu[["station", "day"] + isu_value_cols].rename(
    columns={c: f"isu_{c}" for c in isu_value_cols}
)

df = df.merge(
    df_isu_renamed,
    left_on=["climate_station", "activity_date"],
    right_on=["station", "day"],
    how="left",
)
df = df.drop(columns=["station", "day"])

print(f"Dropped as redundant with PRISM: {ISU_REDUNDANT_WITH_PRISM}")
print(f"Merged shape: {df.shape}")
print(f"ISU climate match rate: {df['isu_climo_high_c'].notna().mean():.1%}")
assert len(df) == len(df_wq), "ISU climate join fanned out WQ rows"

Dropped as redundant with PRISM: ['max_temp_c', 'min_temp_c', 'max_dewpoint_c', 'min_dewpoint_c']
Merged shape: (48251, 79)
ISU climate match rate: 95.2%


## Step 4: USGS streamflow — nearest-gauge spatial match

Same pattern as ISU climate: gauges have no shared key with WQ stations, so
each WQ station is matched to its nearest USGS streamflow gauge by haversine
distance (gauge coordinates come straight from `usgs-iowa-gauges-clean.csv`,
no external fetch needed), then discharge is joined on the matched gauge +
date.

In [8]:
df_gauges = pd.read_csv(f"{CLEAN}/streamflow/usgs-iowa-gauges-clean.csv")
df_discharge = pd.read_csv(f"{CLEAN}/streamflow/usgs-iowa-discharge-clean.csv", parse_dates=["date"])

print(f"Gauges: {df_gauges.shape}")
print(f"Discharge station-days: {df_discharge.shape}")

Gauges: (703, 7)
Discharge station-days: (556850, 4)


In [9]:
gauge_match = nearest_neighbor_match(
    sources=df_stations,
    source_id="MonitoringLocationIdentifier",
    target_lookup=df_gauges,
    target_id="site_no",
    target_lat="latitude",
    target_lon="longitude",
).rename(columns={"site_no": "streamflow_site_no", "distance_km": "distance_to_streamflow_gauge_km"})
gauge_match = gauge_match.merge(
    df_gauges[["site_no", "station_name", "drain_area_sqmi"]].rename(
        columns={"site_no": "streamflow_site_no", "station_name": "streamflow_gauge_name",
                 "drain_area_sqmi": "streamflow_gauge_drain_area_sqmi"}
    ),
    on="streamflow_site_no", how="left",
)

print("Nearest streamflow gauge distance (km):")
print(gauge_match["distance_to_streamflow_gauge_km"].describe().round(2))

df = df.merge(gauge_match, on="MonitoringLocationIdentifier", how="left")
assert len(df) == len(df_wq), "Gauge mapping join fanned out WQ rows"

Nearest streamflow gauge distance (km):
count    1666.00
mean        6.71
std         5.06
min         0.00
25%         2.51
50%         6.06
75%        10.06
max        25.83
Name: distance_to_streamflow_gauge_km, dtype: float64


In [10]:
df_discharge_renamed = df_discharge.rename(
    columns={"site_no": "streamflow_site_no", "discharge_cfs": "streamflow_discharge_cfs",
             "discharge_cd": "streamflow_discharge_cd"}
)

df = df.merge(
    df_discharge_renamed,
    left_on=["streamflow_site_no", "activity_date"],
    right_on=["streamflow_site_no", "date"],
    how="left",
)
df = df.drop(columns=["date"])

print(f"Merged shape: {df.shape}")
print(f"Streamflow discharge match rate: {df['streamflow_discharge_cfs'].notna().mean():.1%}")
assert len(df) == len(df_wq), "Discharge join fanned out WQ rows"

Merged shape: (48251, 85)
Streamflow discharge match rate: 50.0%


## Step 5: Final checks and save

In [11]:
df = df.drop(columns=["activity_date"])

print(f"Final shape: {df.shape}")
print(f"Final columns:  {[c for c in df.columns]}")
assert not df.duplicated(subset=["MonitoringLocationIdentifier", "ActivityStartDateTime"]).any(), (
    "Output grain violated: duplicate (station, timestamp) rows"
)

print("\nMatch rates:")
print(f"  PRISM climate:        {df['prism_tmax_c'].notna().mean():.1%}")
print(f"  ISU climate:          {df['isu_climo_high_c'].notna().mean():.1%}")
print(f"  Streamflow discharge: {df['streamflow_discharge_cfs'].notna().mean():.1%}")

df.head(3)

Final shape: (48251, 84)
Final columns:  ['MonitoringLocationIdentifier', 'ActivityStartDateTime', 'Temperature, water_value', 'Temperature, water_unit', 'Dissolved oxygen (DO)_value', 'Dissolved oxygen (DO)_unit', 'pH_value', 'pH_unit', 'Nitrate_value', 'Nitrate_unit', 'Nitrite_value', 'Nitrite_unit', 'Nitrate + Nitrite_value', 'Nitrate + Nitrite_unit', 'Ammonia-nitrogen_value', 'Ammonia-nitrogen_unit', 'Kjeldahl nitrogen_value', 'Kjeldahl nitrogen_unit', 'Orthophosphate_value', 'Orthophosphate_unit', 'Phosphate-phosphorus_value', 'Phosphate-phosphorus_unit', 'Total Phosphorus, mixed forms_value', 'Total Phosphorus, mixed forms_unit', 'Chloride_value', 'Chloride_unit', 'Sulfate_value', 'Sulfate_unit', 'Specific conductance_value', 'Specific conductance_unit', 'Total dissolved solids_value', 'Total dissolved solids_unit', 'Total suspended solids_value', 'Total suspended solids_unit', 'Turbidity_value', 'Turbidity_unit', 'Escherichia coli_value', 'Escherichia coli_unit', 'Chlorophyll a,

,MonitoringLocationIdentifier,ActivityStartDateTime,"Temperature, water_value","Temperature, water_unit",Dissolved oxygen (DO)_value,Dissolved oxygen (DO)_unit,pH_value,pH_unit,Nitrate_value,Nitrate_unit,...,isu_max_feel_c,isu_max_wind_speed_kts,isu_climo_high_c,isu_climo_low_c,streamflow_site_no,distance_to_streamflow_gauge_km,streamflow_gauge_name,streamflow_gauge_drain_area_sqmi,streamflow_discharge_cfs,streamflow_discharge_cd
0,11NPSWRD_WQX-HTLN_EFMO_DOUS1,2017-07-18 15:00:00,19.820,deg C,8.75,mg/L,8.330,std units,NaN,NaN,...,35.102833,11.000000,29.166667,17.111111,5389400,5.535597,"Bloody Run Creek near Marquette, IA",34.13,30.90,A
1,11NPSWRD_WQX-HTLN_EFMO_DOUS1,2023-07-19 11:00:00,16.372,deg C,12.24,mg/L,8.215,std units,NaN,NaN,...,26.000000,11.296691,29.166667,17.111111,5389400,5.535597,"Bloody Run Creek near Marquette, IA",34.13,22.00,A
2,11NPSWRD_WQX-HTLN_HEHO_HOOV1,2017-07-17 17:00:00,19.920,deg C,7.62,mg/L,8.195,std units,NaN,NaN,...,28.876911,11.000000,30.000000,18.888889,5464942,0.239636,"Hoover Cr at Hoover Nat Hist Site, West Branch...",2.58,1.25,A


In [12]:
os.makedirs(OUT_DIR, exist_ok=True)
df.to_csv(OUT_FILE, index=False)
print(f"Saved {len(df):,} rows x {df.shape[1]} cols -> {OUT_FILE}")

Saved 48,251 rows x 84 cols -> ../../data/03a_merge_primary/wq-daily-environment.csv
